# Volatility Dynamics 15 — Student-t **return** Markov transitions

**Frozen scientific protocol:** [VD 032](../../discussions/Volatility_Dynamics/032_chatgpt_vd15_student_t_return_markov_preregistration.md). **Historical results exploratory only.**

This experiment is about a five-minute **return** transition kernel. It does **not** apply a Student-t law to stochastic-volatility-factor innovations, nor does it fit a fresh Student-t distribution to each terminal horizon.

Compared models: `T_RW` (iid native t returns), `T_MC` (native t AR(1) returns), `AM_G` (unchanged VD14 Markov Gaussian-mixture benchmark), `AM_T` (native Student-t return innovation with stochastic Markov factor paths), `AM_T_AR` (AM_T plus return Markov AR(1)), and `M2_student_t` (VD14 direct horizon-specific Student-t). Seven horizons: 5/10/15/30/60/120/240 minutes, primary 60m. Six assets with sufficient historical data.

Only local `DERIBIT_DATA_ROOT/history/price_5m` price files. No option pricing, no database writes, no cross-branch model imports, no fitted results from VD14. The historical sample has been previously inspected; output is **not independent confirmation**.

## 0. Environment and fixed registration

In [1]:
from pathlib import Path
import os, sys, json, time, warnings
import pandas as pd
from IPython.display import display
from dotenv import load_dotenv
ROOT=Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT/'src'/'data_engineering').exists():
    ROOT=ROOT.parent
assert (ROOT/'src'/'data_engineering').exists(), 'Run from the project repository.'
sys.path.insert(0,str(ROOT/'src'))
load_dotenv(ROOT/'.env')
from data_engineering.modeling import markov_lift_distributions as base
from data_engineering.modeling import student_t_return_markov as mk
DATA=os.getenv('DERIBIT_DATA_ROOT')
assert DATA and Path(DATA).exists(), 'Set DERIBIT_DATA_ROOT to your historical market-data root.'
HIST=Path(DATA)/'history'/'price_5m'
assert HIST.is_dir(), f'Missing price history directory: {HIST}'
OUT=ROOT/'notebooks'/'Volatility_Dynamics'/'15'
OUT.mkdir(parents=True,exist_ok=True)
ASSETS=('AVAX','BTC','ETH','HYPE','SOL','TRX','XRP')
PARTICLES=int(os.getenv('VD15_PARTICLES','256'))
BOOT=int(os.getenv('VD15_BOOTSTRAPS','1000'))
assert PARTICLES==mk.PARTICLES_DEFAULT and BOOT==mk.B_DEFAULT, ('Binding historical protocol requires 256 particles and 1000 resamples. Set both defaults.')
print('VD15',mk.VERSION,'| historical exploratory only | source',HIST)
print(json.dumps(mk.registration(),indent=2,default=str))


VD15 VD15-0.1.0 | historical exploratory only | source D:\MarketData\Deribit\history\price_5m
{
  "version": "VD15-0.1.0",
  "horizons": [
    5,
    10,
    15,
    30,
    60,
    120,
    240
  ],
  "primary_horizon_min": 60,
  "df_grid": [
    3.0,
    4.0,
    6.0,
    10.0,
    20.0,
    40.0
  ],
  "sobol_particles": 256,
  "n_bootstrap": 1000,
  "split": "VD14 60/20/20 chronological UTC days; same valid origins per horizon",
  "historical_end_exclusive": "2026-09-19 00:00:00+00:00",
  "historical_is_exploratory": true,
  "source_module": "markov_lift_distributions.py (same Volatility Dynamics branch)",
  "model_selection": "AR and scale TRAIN; native 5m nu VALID; no TEST tuning",
  "nll_primary": "AM_T vs both AM_G and M2; density and tail calibration jointly"
}


## 1. Verify implementation before loading market data

In [2]:
import subprocess
p=subprocess.run([sys.executable,'-m','pytest','-q',
       str(ROOT/'tests'/'test_student_t_return_markov.py')],
       cwd=str(ROOT),text=True,capture_output=True)
print(p.stdout[-3500:],p.stderr[-1200:])
assert p.returncode==0, 'Synthetic scientific tests must pass before reading real data.'


.....                                                                    [100%]
5 passed in 5.17s
 


## 2. Construct causal native-five-minute data panel

Only `state_time` and `close` are read from external historical CSVs. Returns are native 5m close-to-close and are set to missing at data gaps. Same de-clocking and 30m/240m causal EMA state as VD14; target `R_h` sums future returns within split and rejects missing intermediate bars. Never fit on future targets.

In [3]:
START=time.monotonic()
raw,excluded=base.load_returns(HIST,ASSETS)
print('Excluded:',excluded,'raw observations:',len(raw))
panel=base.panelize(raw)
del raw
features=base.make_features(panel)
frame=base.make_targets(panel,features,mk.HORIZONS)
print(panel.groupby('split').agg(days=('date','nunique'),rows=('return_5m','size'),
        valid_returns=('return_5m','count')).to_string())
print('Causal panel ready; seconds',round(time.monotonic()-START,1))


Excluded: {'HYPE': 'insufficient history'} raw observations: 1292538
       days    rows  valid_returns
split                             
test    150  259200         259200
train   448  774138         774132
valid   150  259200         259200
Causal panel ready; seconds 11.6


## 3. Run six model comparisons at all seven horizons

All six models within a horizon are evaluated at exactly the same forecast origins. Native return Student-t degrees of freedom are selected exclusively on **5m VALID** outcomes and are held fixed at longer horizons. AM-T densities integrate 5m Student-t kernels with Markov factor paths using 256 randomized-Sobol Rao-Blackwell quadrature nodes. Baseline M2 and AM-G are recomputed from the original VD14 model definitions; no output CSV is imported for model fitting.

In [4]:
scores,boot,per_asset,parameters=mk.run_all(frame,particles=PARTICLES,B=BOOT)
assert len(scores)==len(mk.HORIZONS)*len(mk.MODELS)
assert (scores.groupby('horizon_min').n_origins.nunique()==1).all()
print('Scored in',round(time.monotonic()-START,1),'seconds')
print(scores.groupby('horizon_min').agg(models=('model','nunique'),
      origins=('n_origins','first'),days=('n_days','first')).to_string())


Scored in 594.4 seconds
             models  origins  days
horizon_min                       
5                 6   259194   150
10                6   259188   150
15                6   259182   150
30                6   259164   150
60                6   259128   150
120               6   259056   150
240               6   258912   150


## 4. All-horizon scores, primary 60-minute comparisons and asset-level stability

In [5]:
display(scores.round(7))
primary=boot[(boot.horizon_min==mk.PRIMARY)&(boot.metric.isin(['nll','tail_logloss']))]
display(primary.round(7))
display(per_asset[(per_asset.horizon_min==mk.PRIMARY)].round(7))


,horizon_min,model,n_origins,n_days,nll,tail_logloss,tail_brier,tail_observed,tail_predicted,interval_miss95
0,5,T_RW,259194,150,-5.266256,0.096779,0.018101,0.018048,0.037076,0.024943
1,5,T_MC,259194,150,-5.257933,0.097169,0.018118,0.018048,0.038627,0.025001
2,5,AM_G,259194,150,-5.340947,0.078895,0.016791,0.018048,0.012169,0.120265
3,5,AM_T,259194,150,-5.354158,0.078867,0.016741,0.018048,0.013444,0.105006
4,5,AM_T_AR,259194,150,-5.347406,0.078027,0.016731,0.018048,0.014276,0.104856
5,5,M2_student_t,259194,150,-5.081550,0.081984,0.016794,0.018048,0.009748,0.231525
6,10,T_RW,259188,150,-4.884017,0.096713,0.018147,0.018122,0.036777,0.024226
7,10,T_MC,259188,150,-4.888315,0.095921,0.018092,0.018122,0.035173,0.025869
8,10,AM_G,259188,150,-5.007702,0.082709,0.016798,0.018122,0.012498,0.105213
9,10,AM_T,259188,150,-5.017126,0.082215,0.016786,0.018122,0.012939,0.099704


,horizon_min,comparison,metric,delta,q025,q975,test_days,origins
72,60,T_MC - T_RW,nll,-0.011482,-0.013825,-0.008843,150,259128
73,60,T_MC - T_RW,tail_logloss,-0.000758,-0.001543,0.000190,150,259128
75,60,AM_T - AM_G,nll,-0.084547,-0.116395,-0.060888,150,259128
76,60,AM_T - AM_G,tail_logloss,-0.014642,-0.019048,-0.010656,150,259128
78,60,AM_T - M2_student_t,nll,0.067198,0.053084,0.084978,150,259128
79,60,AM_T - M2_student_t,tail_logloss,0.013001,0.008355,0.017629,150,259128
81,60,AM_T_AR - AM_T,nll,0.009213,0.006811,0.011587,150,259128
82,60,AM_T_AR - AM_T,tail_logloss,0.002259,0.001476,0.003117,150,259128
84,60,AM_T_AR - M2_student_t,nll,0.076411,0.062493,0.092475,150,259128
85,60,AM_T_AR - M2_student_t,tail_logloss,0.015260,0.010424,0.020453,150,259128


,horizon_min,asset,model,nll,tail_logloss,interval_miss95,tail_predicted,n_origins
144,60,AVAX,AM_G,-3.602205,0.121371,0.088705,0.010307,43188
145,60,AVAX,M2_student_t,-3.741882,0.085510,0.059646,0.015438,43188
146,60,AVAX,T_RW,-3.554952,0.098415,0.023525,0.037510,43188
147,60,AVAX,T_MC,-3.635447,0.092199,0.041586,0.015871,43188
148,60,AVAX,AM_T,-3.684984,0.102006,0.086922,0.011203,43188
149,60,AVAX,AM_T_AR,-3.596229,0.119105,0.137793,0.005057,43188
150,60,BTC,AM_G,-4.210204,0.144551,0.086575,0.022873,43188
151,60,BTC,M2_student_t,-4.341880,0.120834,0.061962,0.025605,43188
152,60,BTC,T_RW,-4.173458,0.140183,0.039386,0.036501,43188
153,60,BTC,T_MC,-4.177238,0.139985,0.040683,0.034906,43188


## 5. Numerical integration sensitivity — validation only

QMC errors can mimic a small likelihood gain. Compare 256 vs 512 nested integration nodes on a fixed, evenly spaced 60m VALID sample (up to 512 origins per asset); **do not tune the primary test decision** using numerical sensitivity.

In [6]:
particle_check=mk.numerical_sensitivity(frame,parameters['model_kernels'],
                      parameters['volatility_markov_states'],particles=PARTICLES)
display(particle_check.round(8))
print('Numerical diagnostics are descriptive and cannot authorize post-test model changes.')


,asset,model,horizon_min,origins,particles_low,particles_high,nll_delta_high_minus_low,nll_mean_absolute_difference,tail_mean_absolute_difference
0,AVAX,AM_T,60,512,256,512,-0.003528,0.007460,0.000158
1,AVAX,AM_T_AR,60,512,256,512,-0.007760,0.013607,0.000087
2,BTC,AM_T,60,512,256,512,0.003167,0.004460,0.000462
3,BTC,AM_T_AR,60,512,256,512,0.004067,0.006584,0.000910
4,ETH,AM_T,60,512,256,512,-0.005198,0.007370,0.000237
5,ETH,AM_T_AR,60,512,256,512,-0.005447,0.007669,0.000235
6,SOL,AM_T,60,512,256,512,0.002561,0.004113,0.000246
7,SOL,AM_T_AR,60,512,256,512,0.002638,0.004181,0.000242
8,TRX,AM_T,60,512,256,512,0.010121,0.013903,0.000110
9,TRX,AM_T_AR,60,512,256,512,0.009085,0.012849,0.000120


Numerical diagnostics are descriptive and cannot authorize post-test model changes.


## 6. Preregistered decision: exploratory evidence only

In [7]:
result=mk.decision(scores,boot)
print(json.dumps(result,indent=2))
assert result['status']!='CONFIRMED'
assert not result['historical_confirmation_permitted']


{
  "status": "HISTORICAL_EXPLORATORY",
  "n_test_days": 150,
  "primary_horizon_min": 60,
  "flags": {
    "AM_T_beats_AM_G": true,
    "AM_T_beats_M2": false,
    "AM_T_AR_beats_AM_T": false,
    "AM_T_AR_beats_M2": false,
    "T_MC_beats_T_RW": true
  },
  "strong_candidate": false,
  "historical_confirmation_permitted": false,
  "note": "Tail calibration, particle sensitivity, per-asset stability and prospective sample required."
}


## 7. Save compact scored outputs, model parameters and complete manifest

In [8]:
scores.to_csv(OUT/'15_scores.csv',index=False,float_format='%.10g')
boot.to_csv(OUT/'15_bootstrap.csv',index=False,float_format='%.10g')
per_asset.to_csv(OUT/'15_per_asset.csv',index=False,float_format='%.10g')
particle_check.to_csv(OUT/'15_particle_sensitivity.csv',index=False,float_format='%.10g')
(OUT/'15_decision.json').write_text(json.dumps(result,indent=2),encoding='utf-8')
manifest={'module':'Volatility_Dynamics_15','version':mk.VERSION,
          'registration':mk.registration(),'history_dir':str(HIST),
          'assets':sorted(panel.underlying.unique().tolist()),'excluded_assets':excluded,
          'sample_start':str(panel.state_time.min()),'sample_end':str(panel.state_time.max()),
          'split_days':panel.groupby('split').date.nunique().to_dict(),
          'reference_implementation':'VD14 original definitions refit on same splits',
          'parameters':parameters,'particle_sensitivity_validation_only':True,
          'decision':result,'scientific_caveat':'Previously viewed two-year historical data; no prospective claim.'}
(OUT/'15_manifest.json').write_text(json.dumps(manifest,indent=2,default=str),encoding='utf-8')
print('Saved six compact artifacts in',OUT)
print('No origin-level return data, option quotes or DuckDB tables written.')


Saved six compact artifacts in D:\Projects\deribit_market_data\notebooks\Volatility_Dynamics\15
No origin-level return data, option quotes or DuckDB tables written.
